<a href="https://colab.research.google.com/github/yana-g/graph-analysis/blob/main/01_Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Collect Israeli Movies and Actors from Hebrew Wikipedia

This notebook:

1. Retrieves Israeli films from Hebrew Wikipedia categories organized by year.
2. Extracts up to 10 actors from each film page.
3. Cleans invalid labels and table headers that may be mistaken for actor names.
4. Saves a complete movie–actor dataset.
5. Creates an actor-pair edge dataset.
6. Builds and exports a NetworkX collaboration graph.


### 1. Imports

In [1]:
import requests
import pandas as pd
import time
from tqdm.auto import tqdm
from pathlib import Path

API_URL = "https://he.wikipedia.org/w/api.php"

HEADERS = {
    "User-Agent": (
        "IsraeliActorsGraphProject/1.0 "
        "(academic data collection)"
    )
}

session = requests.Session()
session.headers.update(HEADERS)


### 2. Wikipedia API

In [2]:
def wiki_request(params, retries=8):
    """Send a resilient request to the Hebrew Wikipedia API."""
    for attempt in range(retries):
        try:
            response = session.get(
                API_URL,
                params=params,
                timeout=30
            )

            if response.status_code == 200:
                return response.json()

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")

                if retry_after:
                    wait_seconds = int(retry_after)
                else:
                    wait_seconds = min(2 ** attempt, 60)

                print(
                    f"429 received. Waiting "
                    f"{wait_seconds} seconds..."
                )

                time.sleep(wait_seconds)
                continue

            response.raise_for_status()

        except requests.RequestException as exception:
            if attempt == retries - 1:
                raise

            wait_seconds = min(2 ** attempt, 60)

            print(
                f"Request failed: {exception}. "
                f"Waiting {wait_seconds} seconds..."
            )

            time.sleep(wait_seconds)

    raise RuntimeError(
        "Wikipedia API request failed after maximum retries."
    )


### 3. Category Functions

In [3]:
def get_subcategories(category_name):

    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": f"קטגוריה:{category_name}",
        "cmtype": "subcat",
        "cmlimit": "max",
        "format": "json"
    }

    data = wiki_request(params)

    return [
        x["title"].replace("קטגוריה:", "")
        for x in data["query"]["categorymembers"]
    ]

### 4. Movie Collection

In [4]:
def get_movies_from_category(category_name):

    movies = []

    cmcontinue = None

    while True:

        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": f"קטגוריה:{category_name}",
            "cmtype": "page",
            "cmlimit": "max",
            "format": "json"
        }

        if cmcontinue:
            params["cmcontinue"] = cmcontinue

        data = wiki_request(params)

        movies.extend(data["query"]["categorymembers"])

        if "continue" not in data:
            break

        cmcontinue = data["continue"]["cmcontinue"]

        time.sleep(0.2)

    return movies

### 5. Load or Collect Movies

If `movies.csv` already exists, the notebook loads it and skips the Wikipedia category requests.

If the file does not exist, the notebook collects the movie list from Wikipedia and saves it for future runs.

In both cases, the cleaning, validation, actor extraction, graph construction, and summary statistics are recalculated from the loaded data.


In [5]:
MOVIES_FILE = Path("movies.csv")

if MOVIES_FILE.exists():
    print("movies.csv found. Loading the existing movie list...")

    movies_df = pd.read_csv(
        MOVIES_FILE,
        encoding="utf-8-sig"
    )

    data_source = "existing movies.csv"

else:
    print("movies.csv not found. Collecting movies from Wikipedia...")

    year_categories = get_subcategories(
        "סרטים ישראליים לפי שנה"
    )

    print("Year categories:", len(year_categories))

    all_movies = []

    for category in tqdm(year_categories):
        try:
            movies = get_movies_from_category(category)

            year = "".join(
                filter(str.isdigit, category)
            )

            for movie in movies:
                movie["year"] = (
                    int(year)
                    if year
                    else None
                )

            all_movies.extend(movies)

            # Slow down requests to reduce Wikipedia rate limiting.
            time.sleep(0.8)

        except Exception as exception:
            print(
                f"Failed to collect category "
                f"'{category}': {exception}"
            )

            # Wait before continuing to the next category.
            time.sleep(2)

    if not all_movies:
        raise RuntimeError(
            "No movies were collected and movies.csv does not exist."
        )

    movies_df = (
        pd.DataFrame(all_movies)[
            ["pageid", "title", "year"]
        ]
        .rename(columns={
            "pageid": "page_id",
            "title": "movie_title"
        })
    )

    data_source = "Wikipedia API"


movies.csv found. Loading the existing movie list...


In [6]:
# Recalculate movie-list cleaning on every run.
movies_df["year"] = pd.to_numeric(
    movies_df["year"],
    errors="coerce"
)

movies_df["page_id"] = pd.to_numeric(
    movies_df["page_id"],
    errors="coerce"
)

movies_df["movie_title"] = (
    movies_df["movie_title"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

movies_df = (
    movies_df
    .dropna(
        subset=[
            "page_id",
            "movie_title",
            "year"
        ]
    )
    .drop_duplicates(
        subset="page_id"
    )
    .copy()
)

movies_df = movies_df[
    movies_df["movie_title"].ne("")
].copy()

movies_df["page_id"] = (
    movies_df["page_id"]
    .astype(int)
)

movies_df["year"] = (
    movies_df["year"]
    .astype(int)
)

movies_df = (
    movies_df
    .sort_values(
        ["year", "movie_title"]
    )
    .reset_index(drop=True)
)

print("Movie-list source:", data_source)
print("Total clean movies:", len(movies_df))
print(
    "Year range:",
    movies_df["year"].min(),
    "-",
    movies_df["year"].max()
)

display(movies_df.head())


Movie-list source: existing movies.csv
Total clean movies: 1155
Year range: 1960 - 2026


,page_id,movie_title,year
0,1706365,איי לייק מייק (סרט),1960
1,631050,הם היו עשרה,1960
2,1486999,חולות לוהטים,1960
3,2518194,רחל (סרט),1960
4,2504743,תיקון קטן,1960


### 6. Save the Clean Movie List

The cleaned movie list is saved on every run. This keeps `movies.csv` synchronized with the current cleaning rules, even when the original data was loaded from an existing file.


In [7]:
movies_df.to_csv(
    MOVIES_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Saved clean movie list to {MOVIES_FILE}"
)


Saved clean movie list to movies.csv


## Extract and Clean Up to 10 Actors from Each Film Page

The extraction first checks the film infobox and then uses a cast-section fallback.

The cleaning stage removes:

- Wikipedia editing links
- Table headers
- Generic labels
- Empty values
- Numeric-only values
- Duplicate actor names within the same movie

Requests are slowed down and retried to reduce Wikipedia rate-limit errors.


In [8]:
from bs4 import BeautifulSoup
import re
from pathlib import Path
from itertools import combinations
from collections import defaultdict
import networkx as nx
import pickle

MAX_ACTORS = 10
REQUEST_DELAY = 0.6

# A new checkpoint version prevents older, non-normalized metadata
# from being reused automatically.
CHECKPOINT_FILE = Path(
    "movies_actor_metadata_checkpoint.csv"
)


### 7. Page Retrieval

In [9]:
def get_page_html(page_id):
    params = {
        "action": "parse",
        "pageid": int(page_id),
        "prop": "text",
        "format": "json",
        "formatversion": 2
    }
    data = wiki_request(params, retries=7)
    return data["parse"]["text"]


### 8. Actor Extraction Settings

In [25]:
ACTOR_FIELD_LABELS = {
    "שחקנים",
    "שחקנים ראשיים",
    "משתתפים",
    "בכיכוב",
    "בכיכובם",
    "צוות שחקנים",
    "צוות השחקנים",
    "מדבבים",
    "דיבוב"
}

GENRE_FIELD_LABELS = {
    "סוגה",
    "ז'אנר",
    "זאנר",
    "סוג"
}

DIRECTOR_FIELD_LABELS = {
    "בימוי",
    "במאי",
    "במאית"
}

CAST_SECTION_LABELS = {
    "שחקנים",
    "שחקנים ודמויות",
    "דמויות ושחקנים",
    "צוות השחקנים",
    "צוות שחקנים",
    "משתתפים",
    "המשתתפים",
    "דמויות",
    "מדבבים",
    "דיבוב"
}

# Exact labels and non-person entities observed in Wikipedia cast sections.
# The list is intentionally conservative to avoid deleting real actors.
INVALID_ACTOR_NAMES = {
    "IMDb",
    "ויקינתונים",
    "אתר רשמי",
    "עריכת הנתון בוויקינתונים",
    "עריכה",
    "עריכת קוד מקור",
    "קישור",
    "ויקישיתוף",
    "הערות שוליים",
    "שם",
    "שחקן",
    "שחקנית",
    "שחקנים",
    "שחקנ/ית",
    "שחקן/ית",
    "שחקן/שחקנית",
    "שחקן/ת",
    "משתתף/ת",
    "שם שחקן",
    "שם שחקנ/ית",
    "שם השחקנ/ית",
    "שם השחקן",
    "שם השחקנית",
    "שם השחקן/ית",
    "שם השחקן/שחקנית",
    "השחקן / השחקנית",
    "שם הדמות",
    "הדמות",
    "דמות",
    "תפקיד",
    "תפקיד אורח",
    "הערות",
    "ראו פסקת שחקנים",
    "בתי ספר למשחק",
    "ניצול שואה",
    "הבריגדה היהודית",
    "ראו למטה"
}


def normalize_text(value):
    """Normalize text while treating missing values as empty strings."""
    if value is None or pd.isna(value):
        return ""

    value = re.sub(r"\[[^]]*\]", "", str(value))
    value = value.replace("\u00a0", " ")
    value = re.sub(r"\s+", " ", value)

    return value.strip(" ,;:|–—-")


NORMALIZED_INVALID_ACTOR_NAMES = {
    normalize_text(value).casefold()
    for value in INVALID_ACTOR_NAMES
}


def clean_actor_name(name):
    """Return a normalized actor-name string."""
    return normalize_text(name)


def actor_comparison_key(name):
    """
    Return a conservative comparison key.

    Hyphen and whitespace differences are ignored so variants such as
    'בן יעקב' and 'בן-יעקב' can be recognized as the same spelling variant.
    """
    value = clean_actor_name(name).casefold()
    value = re.sub(r"[-–—־]", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def valid_actor_name(name):
    """Check whether a candidate looks like a usable actor name."""
    normalized_name = clean_actor_name(name)

    if not normalized_name:
        return False

    if normalized_name.casefold() in NORMALIZED_INVALID_ACTOR_NAMES:
        return False

    if len(normalized_name) < 2 or len(normalized_name) > 80:
        return False

    if re.fullmatch(r"[0-9]+", normalized_name):
        return False

    if not any(character.isalpha() for character in normalized_name):
        return False

    return True


def unique_first(values, limit=MAX_ACTORS):
    """Clean candidates, remove duplicates, and keep the first values."""
    result = []
    seen = set()

    for value in values:
        cleaned_value = clean_actor_name(value)
        comparison_key = actor_comparison_key(cleaned_value)

        if valid_actor_name(cleaned_value) and comparison_key not in seen:
            seen.add(comparison_key)
            result.append(cleaned_value)

        if len(result) >= limit:
            break

    return result


### 9. Actor Extraction

In [26]:
def actors_from_infobox(soup, max_actors=MAX_ACTORS):
    """Extract actor names from a film infobox."""
    infobox = soup.find(
        "table",
        class_=lambda value: value and "infobox" in value
    )

    if infobox is None:
        return []

    for row in infobox.find_all("tr"):
        header = row.find("th")
        value_cell = row.find("td")

        if header is None or value_cell is None:
            continue

        label = clean_actor_name(
            header.get_text(" ", strip=True)
        )

        if not any(
            field in label
            for field in ACTOR_FIELD_LABELS
        ):
            continue

        candidates = []

        # First, prefer linked names because they are usually cleaner.
        for link in value_cell.find_all("a"):
            candidates.append(
                link.get_text(" ", strip=True)
            )

        actors = unique_first(
            candidates,
            max_actors
        )

        if actors:
            return actors

        # If no usable links exist, split the plain text.
        plain_text = value_cell.get_text("|", strip=True)
        candidates = re.split(
            r"[|,;\n]\s*",
            plain_text
        )

        return unique_first(
            candidates,
            max_actors
        )

    return []


def actors_from_cast_section(soup, max_actors=MAX_ACTORS):
    """Extract actor names from a cast-related page section."""
    for heading in soup.find_all(["h2", "h3", "h4"]):
        title = clean_actor_name(
            heading.get_text(" ", strip=True)
        )

        title = (
            title
            .replace("[עריכת קוד מקור]", "")
            .replace("עריכת קוד מקור", "")
            .replace("עריכה", "")
            .strip()
        )

        if title not in CAST_SECTION_LABELS:
            continue

        candidates = []

        # Read elements until the next section heading.
        for element in heading.find_all_next():
            if (
                element is not heading
                and element.name in ["h2", "h3", "h4"]
            ):
                break

            # Extract the first cell from each cast-table row.
            if element.name == "table":
                for row in element.find_all("tr"):
                    cells = row.find_all(["td", "th"])

                    if not cells:
                        continue

                    actor_cell = cells[0]
                    link = actor_cell.find("a", href=True)

                    if link:
                        candidate = link.get_text(" ", strip=True)
                    else:
                        candidate = actor_cell.get_text(" ", strip=True)

                    candidates.append(candidate)

            # Extract linked names from paragraphs and lists.
            elif element.name in ["p", "ul", "ol", "div"]:
                for link in element.find_all("a", href=True):
                    href = link.get("href", "")

                    if not href.startswith("/wiki/"):
                        continue

                    if any(
                        prefix in href
                        for prefix in [
                            "/wiki/מיוחד:",
                            "/wiki/עזרה:",
                            "/wiki/ויקיפדיה:",
                            "/wiki/קטגוריה:",
                            "/wiki/קובץ:",
                            "/wiki/שיחה:"
                        ]
                    ):
                        continue

                    candidates.append(
                        link.get_text(" ", strip=True)
                    )

        actors = unique_first(
            candidates,
            max_actors
        )

        if actors:
            return actors

    return []


def clean_metadata_value(value):
    """Clean a metadata value extracted from Wikipedia."""
    if value is None:
        return ""

    value = re.sub(r"\[\d+\]", "", str(value))
    value = re.sub(r"\s+", " ", value)

    return value.strip(" ,;|")


def normalize_genre(value):
    """Normalize equivalent Hebrew genre labels."""
    value = clean_metadata_value(value)

    if not value:
        return ""

    # Remove common Wikipedia prefixes from genre values.
    for prefix in ("סרטי ", "סרט "):
        if value.startswith(prefix):
            value = value[len(prefix):].strip()
            break

    genre_mapping = {
        "דוקומנטרי": "תיעודי",
        "סרט תעודה": "תיעודי",
        "קומדיית דרמה": "דרמה קומית",
        "דרמה-קומית": "דרמה קומית"
    }

    return genre_mapping.get(value, value)


def unique_metadata_values(values, normalizer=clean_metadata_value):
    """Normalize values, remove empty entries, and remove duplicates."""
    result = []
    seen = set()

    for value in values:
        cleaned_value = normalizer(value)

        if not cleaned_value:
            continue

        comparison_key = cleaned_value.casefold()

        if comparison_key in seen:
            continue

        seen.add(comparison_key)
        result.append(cleaned_value)

    return result


def metadata_from_infobox(soup):
    """Extract genre and director information from a film infobox."""
    metadata = {
        "genres": [],
        "directors": []
    }

    infobox = soup.find(
        "table",
        class_=lambda value: value and "infobox" in value
    )

    if infobox is None:
        return metadata

    for row in infobox.find_all("tr"):
        header = row.find("th")
        value_cell = row.find("td")

        if header is None or value_cell is None:
            continue

        label = clean_metadata_value(
            header.get_text(" ", strip=True)
        )

        linked_values = [
            link.get_text(" ", strip=True)
            for link in value_cell.find_all("a")
        ]
        linked_values = unique_metadata_values(linked_values)

        if linked_values:
            values = linked_values
        else:
            plain_text = value_cell.get_text("|", strip=True)
            values = unique_metadata_values(
                re.split(r"[|,;\n/]\s*", plain_text)
            )

        if label in GENRE_FIELD_LABELS:
            metadata["genres"] = unique_metadata_values(
                values,
                normalizer=normalize_genre
            )
        elif label in DIRECTOR_FIELD_LABELS:
            metadata["directors"] = unique_metadata_values(
                values
            )

    return metadata


def extract_movie_data(page_html, max_actors=MAX_ACTORS):
    """Extract actors and movie metadata from one Wikipedia film page."""
    soup = BeautifulSoup(
        page_html,
        "html.parser"
    )

    infobox_actors = actors_from_infobox(
        soup,
        max_actors
    )

    section_actors = actors_from_cast_section(
        soup,
        max_actors
    )

    actors = unique_first(
        infobox_actors + section_actors,
        max_actors
    )

    metadata = metadata_from_infobox(soup)

    if not actors:
        source = "not_found"
    elif infobox_actors and section_actors:
        source = "infobox+section"
    elif infobox_actors:
        source = "infobox"
    else:
        source = "section"

    return {
        "actors": actors,
        "source": source,
        "genres": metadata["genres"],
        "directors": metadata["directors"]
    }


### 10. Test the Extraction Before Processing the Full Dataset

The test sample helps verify that the extractor returns real actor names rather than table headers or generic labels.


In [27]:
test_rows = movies_df.sample(min(10, len(movies_df)), random_state=42)
test_results = []

for row in test_rows.itertuples(index=False):
    try:
        html = get_page_html(row.page_id)
        movie_data = extract_movie_data(html)
        test_results.append({
            "movie_title": row.movie_title,
            "year": row.year,
            "actors": movie_data["actors"],
            "genre": " | ".join(movie_data["genres"]),
            "director": " | ".join(movie_data["directors"]),
            "source": movie_data["source"]
        })
        time.sleep(REQUEST_DELAY)
    except Exception as exc:
        test_results.append({"movie_title": row.movie_title, "year": row.year,
                             "actors": [], "source": f"error: {exc}"})

pd.DataFrame(test_results)


,movie_title,year,actors,genre,director,source
0,ליידי טיטי,2018,"[לירית בלבן, צביקה היזיקיאס, אלזה עלמו, תהילה ...",קומדיה,אסתי עלמו וקסלר,infobox+section
1,אחד באפריל (סרט),1989,"[ציפי שביט, ספי ריבלין, נפתלי אלטר, דין זילברמ...",קומדיה,מנחם זילברמן,infobox+section
2,חגיגה לעיניים,1975,"[יוסף שילוח, מוסקו אלקלעי, טליה שפירא, דורי בן...",קומדיה שחורה,אסי דיין,infobox
3,שאנן שיא,1991,[],תיעודי | סטודנטים,ארי פולמן | אורי סיון,not_found
4,"המעורר, י.ח ברנר",2015,[],תיעודי,יאיר קדר,not_found
5,שגעון של אבא,1981,"[גבי עמרני, שושיק שני, ששי קשת, קרוליין לנגפור...",קומדיה,אלי שגיא,infobox+section
6,"בן-גוריון, אפילוג",2016,"[דוד בן-גוריון, קלינטון ביילי, פולה בן-גוריון,...",תיעודי,יריב מוזר,infobox+section
7,תחיה ותהיה,2005,[יעל אבקסיס],דרמה,ראדו מיכאיליאנו,infobox
8,"סאלח, פה זה ארץ ישראל",2017,[ירון לונדון],תיעודי,דוד דרעי,infobox
9,ילדי השמש,2007,[],תיעודי,רן טל,not_found


### 11. Process All Films with Checkpoint Support

The checkpoint allows the notebook to continue from saved progress if Colab disconnects or the run is stopped.

A clean checkpoint filename is used so that results created by an older extraction rule are not reused.


In [28]:
def load_actor_checkpoint():
    """Load previously collected records, when available."""
    if not CHECKPOINT_FILE.exists():
        return {}

    checkpoint_df = pd.read_csv(
        CHECKPOINT_FILE,
        encoding="utf-8-sig"
    )

    return (
        checkpoint_df
        .set_index("page_id")
        .to_dict(orient="index")
    )


def save_actor_checkpoint(records):
    """Save the current collection progress."""
    pd.DataFrame(records).to_csv(
        CHECKPOINT_FILE,
        index=False,
        encoding="utf-8-sig"
    )


def normalize_pipe_separated_values(value, normalizer):
    """Normalize and deduplicate values stored with a pipe separator."""
    if pd.isna(value) or not str(value).strip():
        return ""

    values = [
        item.strip()
        for item in str(value).split("|")
    ]

    normalized_values = unique_metadata_values(
        values,
        normalizer=normalizer
    )

    return " | ".join(normalized_values)


def clean_record_actor_values(record):
    """Recalculate actor and metadata cleaning for one movie record."""
    cleaned_actors = unique_first(
        [
            record.get(f"actor_{index}", "")
            for index in range(1, MAX_ACTORS + 1)
        ],
        MAX_ACTORS
    )

    for index in range(1, MAX_ACTORS + 1):
        record[f"actor_{index}"] = (
            cleaned_actors[index - 1]
            if index <= len(cleaned_actors)
            else ""
        )

    record["actor_count"] = len(cleaned_actors)

    record["genre"] = normalize_pipe_separated_values(
        record.get("genre", ""),
        normalizer=normalize_genre
    )

    record["director"] = normalize_pipe_separated_values(
        record.get("director", ""),
        normalizer=clean_metadata_value
    )

    return record


def collect_actors_for_all_movies(
    movies_dataframe,
    checkpoint_every=25
):
    """Load saved records when possible and download only missing movies."""
    saved = load_actor_checkpoint()
    records = []

    for row in tqdm(
        movies_dataframe.itertuples(index=False),
        total=len(movies_dataframe)
    ):
        page_id = int(row.page_id)

        if page_id in saved:
            saved_record = {
                "page_id": page_id,
                **saved[page_id]
            }

            records.append(
                clean_record_actor_values(saved_record)
            )
            continue

        record = {
            "page_id": page_id,
            "movie_title": row.movie_title,
            "year": row.year,
            "genre": "",
            "director": "",
            "actor_count": 0,
            "extraction_source": "not_processed",
            "error": ""
        }

        for index in range(1, MAX_ACTORS + 1):
            record[f"actor_{index}"] = ""

        try:
            html = get_page_html(page_id)
            movie_data = extract_movie_data(html)

            record["extraction_source"] = movie_data["source"]
            record["genre"] = " | ".join(movie_data["genres"])
            record["director"] = " | ".join(movie_data["directors"])

            for index, actor in enumerate(
                movie_data["actors"],
                start=1
            ):
                record[f"actor_{index}"] = actor

        except Exception as exception:
            record["extraction_source"] = "error"
            record["error"] = str(exception)

        record = clean_record_actor_values(record)
        records.append(record)

        if len(records) % checkpoint_every == 0:
            save_actor_checkpoint(records)

        time.sleep(REQUEST_DELAY)

    save_actor_checkpoint(records)
    return pd.DataFrame(records)


def canonicalize_actor_spellings(dataframe):
    """
    Merge punctuation-only spelling variants across the full dataset.

    For each conservative comparison key, the most frequent observed spelling
    is used as the canonical representation.
    """
    dataframe = dataframe.copy()

    actor_columns = [
        f"actor_{index}"
        for index in range(1, MAX_ACTORS + 1)
    ]

    actor_values = (
        dataframe[actor_columns]
        .stack()
        .map(clean_actor_name)
    )

    actor_values = actor_values[
        actor_values.map(valid_actor_name)
    ]

    counts = actor_values.value_counts()

    variants = defaultdict(list)

    for actor, count in counts.items():
        variants[
            actor_comparison_key(actor)
        ].append((actor, int(count)))

    canonical_map = {}

    for key, options in variants.items():
        options = sorted(
            options,
            key=lambda item: (
                -item[1],
                len(item[0]),
                item[0]
            )
        )

        canonical_name = options[0][0]

        for actor, _ in options:
            canonical_map[actor] = canonical_name

    for column in actor_columns:
        dataframe[column] = dataframe[column].map(
            lambda value: (
                canonical_map.get(
                    clean_actor_name(value),
                    clean_actor_name(value)
                )
                if valid_actor_name(value)
                else ""
            )
        )

    # Remove duplicates that may appear after canonicalization and recalculate count.
    cleaned_records = []

    for record in dataframe.to_dict(orient="records"):
        cleaned_records.append(
            clean_record_actor_values(record)
        )

    return pd.DataFrame(cleaned_records), canonical_map


movies_with_actors_df = collect_actors_for_all_movies(
    movies_df
)

movies_with_actors_df, actor_canonical_map = (
    canonicalize_actor_spellings(
        movies_with_actors_df
    )
)

# Save the cleaned checkpoint so future runs reuse the corrected values.
save_actor_checkpoint(
    movies_with_actors_df.to_dict(orient="records")
)

print(
    "Canonicalized actor spellings:",
    sum(
        original != canonical
        for original, canonical
        in actor_canonical_map.items()
    )
)

movies_with_actors_df.head()


  0%|          | 0/1155 [00:00<?, ?it/s]

Canonicalized actor spellings: 0


,page_id,movie_title,year,genre,director,actor_count,extraction_source,error,actor_1,actor_2,actor_3,actor_4,actor_5,actor_6,actor_7,actor_8,actor_9,actor_10
0,1706365,איי לייק מייק (סרט),1960,קומדיה רומנטית,פיטר פריי,9,infobox+section,NaN,בתיה לנצט,חיים טופול,אילנה רובינא,גדעון זינגר,זאב ברלינסקי,אבנר חזקיהו,גאולה נוני,סיי גיטין,מאירה שור,
1,631050,הם היו עשרה,1960,,ברוך דינר,9,infobox,NaN,עודד תאומי,בומבה צור,ליאו פילר,גבריאל דגן,יהודה גבאי,ניסים עזיקרי,איתן פריבר,ישראל רובינצ'יק,נינט דינר,
2,1486999,חולות לוהטים,1960,פעולה | דרמה,רפאל נוסבאום,9,infobox,NaN,דליה לביא,אורי זוהר,עודד תאומי,עודד קוטלר,גרט הופמן,אנ',נתן כוגן,גילה אלמגור,אסתר עופרים,
3,2518194,רחל (סרט),1960,דרמה | פשע,נורי חביב,5,infobox,NaN,רחל טטרקו (חביב),דוד רם,יצחק בנש,מרגלית שמיט,ששון סער,,,,,
4,2504743,תיקון קטן,1960,דרמה,חיים חפר | דן בן אמוץ,5,section,NaN,אורי זוהר,יהורם גאון,מישא אשרוב,גברי בנאי,"להקת הנח""ל",,,,,


### 12. Validate and Save the Complete Movie–Actor Dataset

This section checks extraction coverage and verifies that invalid labels were not retained as actor names.


In [29]:
actor_columns = [
    f"actor_{index}"
    for index in range(1, MAX_ACTORS + 1)
]

print("Total films:", len(movies_with_actors_df))
print(
    "Films with at least one actor:",
    (movies_with_actors_df["actor_count"] > 0).sum()
)
print(
    "Films without actors found:",
    (movies_with_actors_df["actor_count"] == 0).sum()
)
print(
    "Request errors:",
    (movies_with_actors_df["extraction_source"] == "error").sum()
)

display(
    movies_with_actors_df[
        "extraction_source"
    ].value_counts(dropna=False)
)

actor_values = (
    movies_with_actors_df[actor_columns]
    .fillna("")
    .astype(str)
    .apply(lambda column: column.str.strip())
)

valid_actor_mask = (
    actor_values.ne("")
    & actor_values.apply(
        lambda column:
        ~column.str.casefold().isin(
            NORMALIZED_INVALID_ACTOR_NAMES
        )
    )
)

actual_actor_count = valid_actor_mask.sum(axis=1)

count_mismatches = (
    actual_actor_count
    != movies_with_actors_df["actor_count"]
).sum()

print(
    "Actor-count mismatches:",
    count_mismatches
)

if count_mismatches:
    display(
        movies_with_actors_df.loc[
            actual_actor_count
            != movies_with_actors_df["actor_count"],
            [
                "movie_title",
                "actor_count",
                *actor_columns
            ]
        ].head(20)
    )

    raise ValueError(
        "actor_count does not match the number of stored actor names."
    )

remaining_invalid = sorted({
    value
    for value in (
        movies_with_actors_df[actor_columns]
        .stack()
        .map(clean_actor_name)
    )
    if value.casefold() in NORMALIZED_INVALID_ACTOR_NAMES
})

print(
    "Remaining known invalid actor labels:",
    remaining_invalid
)

if remaining_invalid:
    raise ValueError(
        "Known non-actor labels remain after cleaning."
    )


Total films: 1155
Films with at least one actor: 1005
Films without actors found: 150
Request errors: 0


,count
extraction_source,
infobox+section,711
infobox,263
not_found,149
section,32


Actor-count mismatches: 0
Remaining known invalid actor labels: []


### 13. Validate Collected Actor Names

The following checks look for invalid labels and suspiciously frequent values before the files are saved.


In [30]:
all_actor_values = (
    movies_with_actors_df[actor_columns]
    .stack()
    .astype(str)
    .map(clean_actor_name)
)

invalid_values_found = sorted({
    value
    for value in all_actor_values
    if value.casefold() in NORMALIZED_INVALID_ACTOR_NAMES
})

print("Invalid actor labels found:", invalid_values_found)
print("Unique actor names:", all_actor_values.nunique())

all_actor_values.value_counts().head(20)


Invalid actor labels found: []
Unique actor names: 3175


,count
,3960
גילה אלמגור,35
משה איבגי,33
אלון אבוטבול,33
זאב רווח,33
יוסף שילוח,31
עמוס לביא,30
אסי דיין,30
גבי עמרני,28
גדעון זינגר,24


In [31]:
# Ensure actor_count is numeric
movies_with_actors_df["actor_count"] = pd.to_numeric(
    movies_with_actors_df["actor_count"],
    errors="coerce"
).fillna(0).astype(int)

movies_without_actors = movies_with_actors_df[
    movies_with_actors_df["actor_count"].eq(0)
][["movie_title", "year"]]

print("Movies without actors:", len(movies_without_actors))

if movies_without_actors.empty:
    print("No movies without actors were found.")
else:
    display(
        movies_without_actors.sample(
            n=min(20, len(movies_without_actors)),
            random_state=42
        )
    )

Movies without actors: 150


,movie_title,year
697,דומא (סרט),2011
423,התור באיבן שדאד 2,2000
1012,שפר בדרך להוליווד,2020
710,לאה גולדברג בחמישה בתים,2011
704,הצלמניה (סרט),2011
473,הילדים של ארנה,2003
659,כן המפקדת,2009
1109,חליסה (סרט),2024
678,זוהי סדום,2010
727,7 הסלילים של יונה וולך,2012


In [32]:
movies_with_actors_df["actor_count"].describe()

,actor_count
count,1155.000000
mean,6.571429
std,3.776794
min,0.000000
25%,3.000000
50%,8.000000
75%,10.000000
max,10.000000


In [33]:
movies_with_actors_df["actor_count"].value_counts().sort_index()

,count
actor_count,
0,150
1,43
2,54
3,56
4,54
5,62
6,66
7,59
8,50


In [34]:
movies_with_actors_df.to_csv("movies_with_actors.csv", index=False, encoding="utf-8-sig")
print("Saved movies_with_actors.csv")


Saved movies_with_actors.csv


## Create the Actor-Pair Edge Dataset

Each row represents two unique actors who appeared together in one film. The movie title, production year, and Wikipedia page ID are retained on every edge record.


In [35]:
edge_records = []

for row in movies_with_actors_df.itertuples(index=False):
    actors = unique_first(
        [
            getattr(row, column)
            for column in actor_columns
        ],
        MAX_ACTORS
    )

    for actor_1, actor_2 in combinations(actors, 2):
        edge_records.append({
            "actor_1": actor_1,
            "actor_2": actor_2,
            "movie_title": row.movie_title,
            "year": row.year,
            "genre": getattr(row, "genre", ""),
            "director": getattr(row, "director", ""),
            "page_id": row.page_id
        })

actor_edges_df = pd.DataFrame(
    edge_records,
    columns=[
        "actor_1",
        "actor_2",
        "movie_title",
        "year",
        "genre",
        "director",
        "page_id"
    ]
)

print(
    "Actor-pair film records:",
    len(actor_edges_df)
)

actor_edges_df.head()


Actor-pair film records: 29374


,actor_1,actor_2,movie_title,year,genre,director,page_id
0,בתיה לנצט,חיים טופול,איי לייק מייק (סרט),1960,קומדיה רומנטית,פיטר פריי,1706365
1,בתיה לנצט,אילנה רובינא,איי לייק מייק (סרט),1960,קומדיה רומנטית,פיטר פריי,1706365
2,בתיה לנצט,גדעון זינגר,איי לייק מייק (סרט),1960,קומדיה רומנטית,פיטר פריי,1706365
3,בתיה לנצט,זאב ברלינסקי,איי לייק מייק (סרט),1960,קומדיה רומנטית,פיטר פריי,1706365
4,בתיה לנצט,אבנר חזקיהו,איי לייק מייק (סרט),1960,קומדיה רומנטית,פיטר פריי,1706365


In [36]:
print(
    "Self loops:",
    (
        actor_edges_df["actor_1"]
        == actor_edges_df["actor_2"]
    ).sum()
)

print(
    "Missing actor names:",
    actor_edges_df[
        ["actor_1", "actor_2"]
    ].isna().sum().sum()
)

invalid_edge_values = set(
    actor_edges_df["actor_1"]
).union(
    actor_edges_df["actor_2"]
)

invalid_edge_values = sorted({
    value
    for value in invalid_edge_values
    if clean_actor_name(value).casefold()
    in NORMALIZED_INVALID_ACTOR_NAMES
})

print(
    "Invalid labels in edges:",
    invalid_edge_values
)


Self loops: 0
Missing actor names: 0
Invalid labels in edges: []


In [37]:
actor_edges_df.to_csv("actor_edges.csv", index=False, encoding="utf-8-sig")
print("Saved actor_edges.csv")


Saved actor_edges.csv


### 14. Build the NetworkX Graph

Each actor is represented by one node.

An undirected edge connects two actors who appeared in the same film. If the same pair collaborated in several films, the edge stores all film titles, years, page IDs, and a collaboration weight.


In [38]:
G = nx.Graph()

# Add every cleaned actor as a node, including actors
# who do not have a collaboration edge.
for row in movies_with_actors_df.itertuples(index=False):
    actors = unique_first(
        [
            getattr(row, column)
            for column in actor_columns
        ],
        MAX_ACTORS
    )

    G.add_nodes_from(actors)

# Add one edge for each unique actor pair.
for row in actor_edges_df.itertuples(index=False):
    actor_1 = row.actor_1
    actor_2 = row.actor_2

    if G.has_edge(actor_1, actor_2):
        G[actor_1][actor_2]["movies"].append(
            row.movie_title
        )
        G[actor_1][actor_2]["years"].append(
            int(row.year)
        )
        G[actor_1][actor_2]["page_ids"].append(
            int(row.page_id)
        )
        G[actor_1][actor_2]["weight"] += 1
    else:
        G.add_edge(
            actor_1,
            actor_2,
            movies=[row.movie_title],
            years=[int(row.year)],
            page_ids=[int(row.page_id)],
            weight=1
        )

print("Actors (nodes):", G.number_of_nodes())
print("Collaborations (edges):", G.number_of_edges())
print(
    "Isolated actors:",
    nx.number_of_isolates(G)
)


Actors (nodes): 3174
Collaborations (edges): 27696
Isolated actors: 18


In [39]:
# GEXF does not support Python lists as edge attributes,
# so list values are converted to strings in a separate copy.
G_gexf = G.copy()

for _, _, data in G_gexf.edges(data=True):
    data["movies"] = " | ".join(
        map(str, data["movies"])
    )
    data["years"] = " | ".join(
        map(str, data["years"])
    )
    data["page_ids"] = " | ".join(
        map(str, data["page_ids"])
    )

nx.write_gexf(
    G_gexf,
    "israeli_actors_graph.gexf"
)

with open(
    "israeli_actors_graph.pkl",
    "wb"
) as file:
    pickle.dump(G, file)

print("Saved israeli_actors_graph.gexf")
print("Saved israeli_actors_graph.pkl")


Saved israeli_actors_graph.gexf
Saved israeli_actors_graph.pkl
